# Assignment 2
## P+7 (Oulipian language modelling)

### Background

The Oulipo (*Ouvroir de Littérature Potentielle*, or “Workshop of Potential Literature”) is a French literary group founded in 1960 by writer Raymond Queneau and mathematician François Le Lionnais. The group focuses on using rules and constraints in writing as a way to spark creativity. Rather than seeing constraints as obstacles, Oulipians treat them as tools to inspire new forms of storytelling and poetry. Their work combines mathematics, language, and playfulness, making their approach both unique and influential in modern literature.

One of the most famous Oulipian writers is Georges Perec, who is known for his creative use of constraints. His novel [*La Disparition* (“A Void”) is written entirely without the letter "e,"](https://archive.org/details/void00pere/mode/2up) which is especially challenging given how common "e" is in French. Perec’s writing often plays with the structure of language in surprising ways.



![George Perec](https://upload.wikimedia.org/wikipedia/commons/7/76/Myart_georges-perec_1978.jpg)

(George Perec, 1978. From Wikidata)

<!-- <img src="https://media.vigliensoni.com/clips/CART498/perec-01.jpg" width="800"> -->


One popular Oulipian technique is N+7, where each noun in a text is replaced by the noun seven entries later in a dictionary. This creates unusual, absurd, and often funny results, encouraging writers to think differently about language and meaning.

## The N+7 technique

One popular Oulipian technique is N+7, where each noun in a text is replaced by the noun seven entries later in a dictionary. This creates unusual, absurd, and often funny results, encouraging writers to think differently about language and meaning.

The N+7 technique process is straightforward

- **Start with a text**. Choose any text—this could be a poem, a sentence, or a passage.
- **Use a dictionary**. Have a dictionary (or word list) handy.
- **Replace each noun**. For every substantive noun in the original text, replace it with the noun appearing seven nouns away in the dictionary. If the end of the dictionary is reached, you can loop back to the beginning.
- **Maintain grammar**. Ensure the new text maintains grammatical correctness as much as possible, though the results often turn out  surreal and nonsensical.

For example, using the N+7 technique with a standard English dictionary for the original sentence:

*One must have a mind of winter*.

- ”mind” → 7 nouns after “mind” is “minimum” or “mindset” (depending on the dictionary)
- “winter” → 7 nouns after  “wishing” or “wipe”

Then, after applying the technique the results are:

- Original: “One must have a mind of winter”
- Dictionary one: “One must have a minimum of wishing”
- Dictionary two: “One must have a mindset of wipe”

[Online example](http://www.spoonbill.org/n+7/)

### Assignment and deliverables

For this assignment, you will create a variation of the N+7 technique we will name P+7. Using the GPT-2 language model, you will replace the last word of each line from *The Snow Man* with the token that has the seventh-highest probability according to the model’s predictions.

By the end of this assignment, submit a link to a GitHub repository named `CART498-GenAI` containing a folder labelled `A02` with the following items:

- A version of the text processed with your P+7 technique, saved as a `.txt` file.
- A second version of the text processed with `P+x`. Choose an `x` value that produces the funniest, wittiest, or most absurd version of the original text. Save this as a .txt file, and include the `x` value in the filename (e.g., `P+23.txt` or `P+12.txt`).
- A Python notebook with the script used to generate your P+7 and P+x transformations using the GPT-2 language model.
- A short reflection (250–350 words) explaining how altering the `x` value impacted the output of your P+x version. Additionally, discuss how you would implement a P+7 technique in which all nouns are replaced with their seventh-highest probability alternatives.

### *The Snow Man*
by Wallace Stevens (1879-1955)


> One must have a mind of winter  
> To regard the frost and the boughs  
> Of the pine-trees crusted with snow;  
> And have been cold a long time  
> To behold the junipers shagged with ice,  
> The spruces rough in the distant glitter  
> Of the January sun; and not to think  
> Of any misery in the sound of the wind,  
> In the sound of a few leaves,  
> Which is the sound of the land  
> Full of the same wind  
> That is blowing in the same bare place  
> For the listener, who listens in the snow,  
> And, nothing himself, beholds  
> Nothing that is not there and the nothing that is.  




In [3]:
pip install transformers torch

In [9]:
from google.colab import drive
drive.mount('/content/drive')

# Helper function to write content to a file in Google Drive
def upload_txt_to_gcs(filename, content):
    try:
        # Specify the path in your Google Drive. Adjust if you have a specific folder.
        file_path = f"/content/drive/MyDrive/{filename}"
        with open(file_path, 'w') as f:
            f.write(content)
        print(f"File '{filename}' successfully written to Google Drive at {file_path}")
    except Exception as e:
        print(f"Error writing file to Google Drive: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re

# -----------------------------
# SETUP
# -----------------------------

tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")
model.eval()

poem = """
One must have a mind of winter
To regard the frost and the boughs
Of the pine-trees crusted with snow;
And have been cold a long time
To behold the junipers shagged with ice,
The spruces rough in the distant glitter
Of the January sun; and not to think
Of any misery in the sound of the wind,
In the sound of a few leaves,
Which is the sound of the land
Full of the same wind
That is blowing in the same bare place
For the listener, who listens in the snow,
And, nothing himself, beholds
Nothing that is not there and the nothing that is.
"""

lines = poem.strip().splitlines()


# -----------------------------
# GET P+X WORD
# -----------------------------

def get_p_x_word(context, x):
    input_ids = tokenizer.encode(context, return_tensors="pt")

    with torch.no_grad():
        logits = model(input_ids).logits[0, -1]

    probabilities = torch.softmax(logits, dim=-1)

    # Get extra predictions because some may be symbols
    _, indices = torch.topk(
        probabilities,
        k=max(x + 50, 100)
    )

    valid_words = []
    seen = set()

    for index in indices:
        token = tokenizer.decode([index]).strip()

        # Only keep tokens made entirely of letters
        if token.isalpha() and token not in seen:
            valid_words.append(token)
            seen.add(token)

        # Once we reach the Xth valid word, return it
        if len(valid_words) == x:
            return valid_words[x - 1]

    return "[NOT_FOUND]"


# -----------------------------
# APPLY P+X TO POEM
# -----------------------------

def apply_p_x(lines, x):

    # P+0 = original poem
    if x == 0:
        return "\n".join(lines)

    result = []

    for line in lines:
        match = re.match(
            r"^(.*?)(\s*)([\w'-]+)([.,;!?:'\"-]*)$",
            line.strip()
        )

        if not match:
            result.append(line)
            continue

        before, space, last_word, punctuation = match.groups()

        # Give GPT-2 everything except the original last word
        context = before + space

        # Find the Xth most probable valid word
        replacement = get_p_x_word(context, x)

        # Replace the original last word
        result.append(
            before + space + replacement + punctuation
        )

    return "\n".join(result)


# -----------------------------
# GENERATE VERSIONS
# -----------------------------

for x in [0, 7, 14]:
    new_poem = apply_p_x(lines, x)

    print(f"\n--- P+{x} Version ---")
    print(new_poem)

    upload_txt_to_gcs(f"P+{x}.txt", new_poem)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


--- P+0 Version ---
One must have a mind of winter
To regard the frost and the boughs
Of the pine-trees crusted with snow;
And have been cold a long time
To behold the junipers shagged with ice,
The spruces rough in the distant glitter
Of the January sun; and not to think
Of any misery in the sound of the wind,
In the sound of a few leaves,
Which is the sound of the land
Full of the same wind
That is blowing in the same bare place
For the listener, who listens in the snow,
And, nothing himself, beholds
Nothing that is not there and the nothing that is.
File 'P+0.txt' successfully written to Google Drive at /content/drive/MyDrive/P+0.txt

--- P+7 Version ---
One must have a mind of iced
To regard the frost and the ills
Of the pine-trees crusted with ik;
And have been cold a long ich
To behold the junipers shagged with ips,
The spruces rough in the distant ers
Of the January sun; and not to ers
Of any misery in the sound of the ich,
In the sound of a few ips,
Which is the sound of the u